In [1]:
import json
import time
import math
import random

import cv2
import numpy as np
import mediapipe as mp
import tensorflow as tf
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

I0000 00:00:1795799372.725167    4969 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795799372.726757    4969 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1795799372.865694    4969 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795799378.033036    4969 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

In [2]:
LANDMARKER_PATH = "face_landmarker.task"
MODEL_PATH = "angry_cnn_best.keras"
CONFIG_PATH = "config.json"
CAMERA_ID = 0

IRIS_L = 468
IRIS_R = 473
NOSE_TIP = 1

BEAM_OPENING_DEG = 20
BEAM_LENGTH = 10

YAW_GAIN = 2.5
PITCH_GAIN = 2.5
ANGLE_LIMIT = 0.5
SMOOTH = 0.3

BEAM_WIDTH = 0.12
FLARE_RADIUS = 0.22
FLICKER = 0.15
GLOW_SIGMA_REL = 0.2
GLOW_STRENGTH = 1.0

RISE_SPEED = 0.2
FALL_SPEED = 0.1

BEAM_LAYERS = [
    (1.00, (0, 90, 255)),
    (0.60, (0, 200, 255)),
    (0.25, (255, 255, 255)),
]
FLARE_LAYERS = [
    (1.00, (0, 120, 255)),
    (0.60, (0, 220, 255)),
    (0.30, (255, 255, 255)),
]

INPUT_SIZE = (48, 48)
CROP_MARGIN = 0.10
INFER_EVERY = 2
PROB_SMOOTH = 0.35
HYSTERESIS = 0.10

EXPLOSION_VIDEO = "explosion.mp4"
EXPLOSION_MASK = "mascara.mkv"

In [3]:
def create_detector():
    base_options = python.BaseOptions(model_asset_path=LANDMARKER_PATH)
    options =  vision.FaceLandmarkerOptions(
        base_options = base_options,
        running_mode = vision.RunningMode.VIDEO,
        num_faces = 1
    )
    return vision.FaceLandmarker.create_from_options(options)

In [4]:
def get_face_points(result, w, h):
    if not result.face_landmarks:
        return None

    points = result.face_landmarks[0]

    left = points[IRIS_L]
    right = points[IRIS_R]
    nose = points[NOSE_TIP]
    
    if right.x < left.x:
        left, right = right, left 

    left_px = (left.x * w, left.y * h)
    right_px = (right.x * w, right.y * h)
    nose_px = (nose.x * w, nose.y * h)

    return left_px, right_px, nose_px

In [5]:
def get_face_box(result, w, h):
    if not result.face_landmarks:
        return None
    pts = np.array([(lm.x * w, lm.y * h) for lm in result.face_landmarks[0]])
    x_min, y_min = pts.min(axis=0)
    x_max, y_max = pts.max(axis=0)
    return x_min, y_min, x_max, y_max

In [6]:
def crop_face(frame, box):
    h, w = frame.shape[:2]
    x_min, y_min, x_max, y_max = box
    cx, cy = (x_min + x_max) / 2, (y_min + y_max) / 2
    side = max(x_max - x_min, y_max - y_min) * (1 + 2 * CROP_MARGIN)
    x0, x1 = int(max(cx - side / 2, 0)), int(min(cx + side / 2, w))
    y0, y1 = int(max(cy - side / 2, 0)), int(min(cy + side / 2, h))
    if x1 - x0 < 10 or y1 - y0 < 10:
        return None, None
    return frame[y0:y1, x0:x1], (x0, y0, x1, y1)

In [7]:
def predict_angry(model, face_bgr):
    gray = cv2.cvtColor(face_bgr, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, INPUT_SIZE, interpolation=cv2.INTER_AREA)
    x = gray.astype("float32")[None, :, :, None]
    prob = float(model(x, training=False)[0, 0])
    return prob, gray

In [8]:
class Explosion:

    def __init__(self, video_path, mask_path, max_width=480):
        cap_v = cv2.VideoCapture(video_path)
        cap_m = cv2.VideoCapture(mask_path)
        if not cap_v.isOpened() or not cap_m.isOpened():
            raise FileNotFoundError("No se pudo abrir el vídeo o la máscara")

        self.fps = cap_v.get(cv2.CAP_PROP_FPS) or 25.0
        self.frames, self.masks = [], []

        while True:
            ok_v, f = cap_v.read()
            ok_m, m = cap_m.read()
            if not (ok_v and ok_m):
                break
            if f.shape[1] > max_width:
                s = max_width / f.shape[1]
                f = cv2.resize(f, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
            m = cv2.cvtColor(m, cv2.COLOR_BGR2GRAY)  # OpenCV la entrega con 3 canales
            m = cv2.resize(m, (f.shape[1], f.shape[0]), interpolation=cv2.INTER_AREA)
            self.frames.append(f)
            self.masks.append(m)

        cap_v.release()
        cap_m.release()
        if not self.frames:
            raise ValueError("El vídeo o la máscara no tienen fotogramas")

        self.t_start = None

    @property
    def active(self):
        return self.t_start is not None

    def play(self):
        if not self.active:
            self.t_start = time.perf_counter()

    def draw_fullscreen(self, frame):
        if not self.active:
            return
        idx = int((time.perf_counter() - self.t_start) * self.fps)
        if idx >= len(self.frames):
            self.t_start = None
            return

        h, w = frame.shape[:2]
        fg = cv2.resize(self.frames[idx], (w, h), interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(self.masks[idx], (w, h), interpolation=cv2.INTER_LINEAR)
        alpha = (mask.astype(np.float32) / 255.0)[:, :, None]
        frame[:] = (fg * alpha + frame * (1.0 - alpha)).astype(np.uint8)

In [9]:
def to_int(p):
    return(int(round(p[0])), int(round(p[1])))

In [10]:
def beam_end(eye, angle, length):
    return (
        eye[0] + length * math.cos(angle),
        eye[1] + length * math.sin(angle)
    )

In [11]:
def clamp(x, lo, hi):
    return max(lo, min(hi, x))

In [12]:
def face_to_img_angle(dx, dy, theta):
    u = (math.cos(theta), math.sin(theta))
    v = (-math.sin(theta), math.cos(theta))
    ix = dx * u[0] + dy * v[0]
    iy = dx * u[1] + dy * v[1]
    return math.atan2(iy, ix)

In [13]:
def draw_beams(frame, beams, eye_mod, intensity):
    h, w = frame.shape[:2]
    layer = np.zeros_like(frame)

    flicker = random.uniform(1 - FLICKER, 1 + FLICKER)
    width = eye_mod * BEAM_WIDTH * flicker

    for eye, end in beams:
        tip = (
            eye[0] + (end[0] - eye[0]) * intensity,
            eye[1] + (end[1] - eye[1]) * intensity
        )
        p0, p1 = to_int(eye), to_int(tip)

        for scale, color in BEAM_LAYERS:
            t = max(1, int(round(width * scale)))
            cv2.line(layer, p0, p1, color, t, cv2.LINE_AA)

        for scale, color in FLARE_LAYERS:
            rad = max(1, int(round(eye_mod * FLARE_RADIUS * flicker * scale)))
            cv2.circle(layer, p0, rad, color, -1, cv2.LINE_AA)

    layer = cv2.convertScaleAbs(layer, alpha=intensity)

    small = cv2.resize(layer, (w // 2, h // 2), interpolation=cv2.INTER_AREA)
    sigma = max(2.0, eye_mod * GLOW_SIGMA_REL / 2)
    small = cv2.GaussianBlur(small, (0, 0), sigma)
    glow = cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR)
    glow = cv2.convertScaleAbs(glow, alpha=GLOW_STRENGTH)

    out = cv2.add(frame, glow)
    out = cv2.add(out, layer)
    return out

In [14]:
model = tf.keras.models.load_model(MODEL_PATH)
with open(CONFIG_PATH) as f:
    umbral = float(json.load(f)["umbral"])
model(np.zeros((1, *INPUT_SIZE, 1), dtype="float32"), training=False)

detector = create_detector()

v = cv2.VideoCapture(CAMERA_ID)

t_0 = time.perf_counter()

smooth_yaw = None
smooth_pitch = None
yaw_0 = 0.0
pitch_0 = 0.0

calibrated = False

intensity = 0.0
last_beams = None

prob_smooth = None
angry = False
frame_idx = 0

explosion = Explosion(EXPLOSION_VIDEO, EXPLOSION_MASK)

while True:
    ret, frame = v.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    h, w = frame.shape[:2]

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

    timestamp = int((time.perf_counter() - t_0) * 1000)
    r = detector.detect_for_video(mp_img, timestamp)

    face = get_face_points(r, w, h)
    box = get_face_box(r, w, h)

    face_ok = face is not None
    if face_ok:
        eye_l, eye_r, nose = face
        
        eye_v = (eye_r[0] - eye_l[0], eye_r[1] - eye_l[1])
        eye_mod = math.hypot(eye_v[0], eye_v[1])
        theta = math.atan2(eye_v[1], eye_v[0])

        u = (math.cos(theta), math.sin(theta))
        v_axis = (-math.sin(theta), math.cos(theta))

        mid = ((eye_l[0] + eye_r[0])/2, (eye_l[1]+eye_r[1])/2)
        nose_v = (nose[0] - mid[0], nose[1] - mid[1])
        yaw = (nose_v[0] * u[0] + nose_v[1] * u[1]) / eye_mod
        pitch = (nose_v[0] * v_axis[0] + nose_v[1] * v_axis[1]) / eye_mod

        if smooth_yaw is None:
            smooth_yaw, smooth_pitch = yaw, pitch
        else:
            smooth_yaw += SMOOTH * (yaw - smooth_yaw)
            smooth_pitch += SMOOTH * (pitch - smooth_pitch)

        yaw = clamp(smooth_yaw - yaw_0, -ANGLE_LIMIT, ANGLE_LIMIT)
        pitch = clamp(smooth_pitch - pitch_0, -ANGLE_LIMIT, ANGLE_LIMIT)

        phi = math.radians(BEAM_OPENING_DEG)
        shift_x = YAW_GAIN * yaw
        shift_y = PITCH_GAIN * pitch

        dir_l = (-math.sin(phi) + shift_x, math.cos(phi) + shift_y)
        dir_r = (math.sin(phi) + shift_x, math.cos(phi) + shift_y)
 
        angle_l = face_to_img_angle(dir_l[0], dir_l[1], theta)
        angle_r = face_to_img_angle(dir_r[0], dir_r[1], theta)

        length = BEAM_LENGTH * eye_mod

        end_l = beam_end(eye_l, angle_l, length)
        end_r = beam_end(eye_r, angle_r, length)

        last_beams = ([(eye_l, end_l), (eye_r, end_r)], eye_mod)

    else:
        smooth_yaw = None
        smooth_pitch = None

    if box is None:
        prob_smooth = None
        angry = False
    else:
        face_img, crop_box = crop_face(frame, box)

        if face_img is not None and frame_idx % INFER_EVERY == 0:
            prob, gray = predict_angry(model, face_img)

            prob_smooth = prob if prob_smooth is None else (
                    prob_smooth + PROB_SMOOTH * (prob - prob_smooth))

            if not angry and prob_smooth >= umbral:
                angry = True
                explosion.play()
            elif angry and prob_smooth < umbral - HYSTERESIS:
                angry = False

        frame_idx += 1

    if face_ok:
        intensity = min(1.0, intensity + RISE_SPEED)
    else:
        intensity = max(0, intensity - FALL_SPEED)

    if intensity > 0 and last_beams is not None:
        frame = draw_beams(frame, last_beams[0], last_beams[1], intensity)

    explosion.draw_fullscreen(frame)

    if face_ok and not calibrated:
        cv2.putText(
            frame, "Mira de frente y pulsa C para calibrar", (20, h - 20),
            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 200, 255), 2
        )
    elif not face_ok:
        cv2.putText(
            frame, "Sin cara", (20, 60),
            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2
        )


    cv2.imshow("Filtro", frame)

    key = cv2.waitKey(1) & 0xFF
    if key in (ord("q"), 27):
            break
    elif key == ord("c") and face_ok:
        yaw_0 = smooth_yaw
        pitch_0 = smooth_pitch
        calibrated = True
    
v.release()
cv2.destroyAllWindows()
detector.close()

E0000 00:00:1795799382.238324    4969 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
W0000 00:00:1795799383.513288    5094 face_landmarker_graph.cc:181] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1795799383.534230    5098 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1795799383.572521    5098 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
